# Train on Colab (GPU)

Runs `scripts/train.py` on a Colab GPU and saves the log, the new `results.csv` rows and the checkpoint to Google Drive (`MyDrive/tiger-reid-runs/`). Uses the same frozen `splits/splits_v1.csv` as local runs.

**One-time setup**
1. *Runtime → Change runtime type → T4 GPU*.
2. The repo is private. Create a GitHub **fine-grained token**: only `Adi-Kaul/tiger-reid`, permission *Contents: Read-only*.
3. In Colab, click the 🔑 *Secrets* icon in the left sidebar → add `GH_TOKEN` = the token → turn on notebook access.

Then *Runtime → Run all*. To run a different model, change `CONFIG` / `SEED` in the settings cell.

## Settings

In [ ]:
CONFIG = "configs/dinov2s_336.yaml"   # dinov2b.yaml, dinov2s_reg.yaml, dinov2s_arcface.yaml, resnet50_arcface.yaml
SEED = 0
EXTRA_ARGS = ""          # e.g. "--epochs 5" for a quick test; "--eval-test" only for final configs

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Mount Drive now (not after training) so the permission pop-up doesn't time out while you're away
from google.colab import drive
drive.mount("/content/drive")

## Code + environment

In [ ]:
import os, subprocess
from google.colab import userdata

REPO = "/content/tiger-reid"
URL = "https://github.com/Adi-Kaul/tiger-reid.git"
# The token goes in through an env var + credential helper, never in the command line,
# so it can't show up in error messages or saved notebook output.
env = {**os.environ, "GH_TOKEN": userdata.get("GH_TOKEN")}
auth = ["-c", "credential.helper=", "-c",
        "credential.helper=!f() { echo username=x-access-token; echo password=$GH_TOKEN; }; f"]

def git(*args):
    r = subprocess.run(["git", *auth, *args], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git {args[0]} failed:\n{r.stderr}")

if not os.path.exists(REPO):
    git("clone", "-q", URL, REPO)
else:
    # Re-used runtime: discard local edits from the last run (results are already in Drive)
    # and jump to the latest main. data/ and checkpoints/ are gitignored, so they stay.
    git("-C", REPO, "fetch", "-q", "origin", "main")
    git("-C", REPO, "reset", "-q", "--hard", "origin/main")
%cd {REPO}
!git log --oneline -1

In [ ]:
# torch / torchvision / numpy / pandas / opencv are preinstalled on Colab.
# Pin the libraries our code calls directly to the versions used locally (newer pml renamed sampler args).
!pip install -q timm==1.0.30 pytorch-metric-learning==2.9.0 pyyaml tabulate
!pip install -q --no-deps -e .
import timm, pytorch_metric_learning, torch
print("torch", torch.__version__, "| timm", timm.__version__, "| pml", pytorch_metric_learning.__version__)

In [ ]:
!python scripts/download_atrw.py   # ATRW re-ID, ~220 MB, ~1 min

In [ ]:
!python -m pytest -q 2>&1 | tail -3   # should be all passed

## Train

In [ ]:
import yaml
name = yaml.safe_load(open(CONFIG))["name"]
run = f"{name}_s{SEED}"
n_rows_before = sum(1 for _ in open("results/results.csv"))
!python scripts/train.py --config {CONFIG} --seed {SEED} --num-workers 2 {EXTRA_ARGS} 2>&1 | tee run.log

## Save to Google Drive

In [ ]:
import shutil, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")   # no-op if already mounted

out = Path(f"/content/drive/MyDrive/tiger-reid-runs/{run}_{time.strftime('%Y%m%d-%H%M')}")
out.mkdir(parents=True, exist_ok=True)
lines = open("results/results.csv").read().splitlines(keepends=True)
(out / "new_rows.csv").write_text(lines[0] + "".join(lines[n_rows_before:]))
shutil.copy("run.log", out)
for src, dst in [(f"results/train_logs/{run}.csv", "train_log.csv"), (f"checkpoints/{run}.pt", f"{run}.pt")]:
    if Path(src).exists():
        shutil.copy(src, out / dst)
    else:
        print(f"MISSING {src}: training failed, see run.log")
print("saved ->", out)
!ls -la "{out}"
print(open(out / "new_rows.csv").read())